# Lesson 1: keyword Search

In [25]:
import warnings

# Disable all warnings
warnings.filterwarnings("ignore")

import os
import weaviate
from dotenv import load_dotenv

load_dotenv()
# This is a public Server
os.environ['WEAVIATE_API_URL'] = 'https://yvi3vi7ytcqjshbpzpeta.c0.us-west3.gcp.weaviate.cloud/'
os.environ['WEAVIATE_API_KEY'] = 'Qlh3QzlUa0M3ZEtKelprcV9VSTBlRHcwRU1YZUVrOU5TZ3dxMENGWXMvWHMreDBDRHlJYWpjd3lpQWEwPV92MjAw'

auth_config = weaviate.auth.AuthApiKey(
    api_key=os.getenv("WEAVIATE_API_KEY")
)

In [26]:
client = weaviate.Client(
    url=os.environ['WEAVIATE_API_URL'],
    auth_client_secret=auth_config,
    additional_headers={
        "X-Cohere-Api-Key": os.environ['COHERE_API_KEY'],
    }
)


In [27]:
client.is_ready() 

True

In [28]:
# Create schema for BM25 (no embeddings)
articles_class = {
    "class": "Articles",
    "description": "News and wiki articles",
    "vectorizer": "none",  # important for keyword/BM25-only search
    "properties": [
        {"name": "title", "dataType": ["text"]},
        {"name": "url",   "dataType": ["text"]},
        {"name": "text",  "dataType": ["text"]},
        {"name": "views", "dataType": ["int"]},
        {"name": "lang",  "dataType": ["text"]},
    ],
    # Optional: tune BM25
    # "invertedIndexConfig": {"bm25": {"k1": 1.2, "b": 0.75}}
}

existing = [c["class"] for c in client.schema.get().get("classes", [])]
if "Articles" not in existing:
    client.schema.create_class(articles_class)

print(client.schema.get())

{'classes': [{'class': 'Articles', 'description': 'News and wiki articles', 'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2}, 'cleanupIntervalSeconds': 60, 'stopwords': {'additions': None, 'preset': 'en', 'removals': None}, 'usingBlockMaxWAND': True}, 'multiTenancyConfig': {'autoTenantActivation': False, 'autoTenantCreation': False, 'enabled': False}, 'properties': [{'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': True, 'name': 'title', 'tokenization': 'word'}, {'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': True, 'name': 'url', 'tokenization': 'word'}, {'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': True, 'name': 'text', 'tokenization': 'word'}, {'dataType': ['int'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': False, 'name': 'views'}, {'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'ind

In [58]:
# Seed multilingual docs for BM25 keyword search
docs_multilang = [
    # English
    {
        "title": "FIFA World Cup Final",
        "url": "https://example.com/fifa-final-en",
        "text": "The FIFA World Cup final is one of the most watched televised events worldwide.",
        "views": 150000000,
        "lang": "en",
    },
    # German
    {
        "title": "FIFA-Weltmeisterschaftsfinale",
        "url": "https://example.com/fifa-final-de",
        "text": "Das Finale der FIFA-Weltmeisterschaft ist eines der meistgesehenen Fernsehereignisse weltweit.",
        "views": 150000000,
        "lang": "de",
    },
    # French
    {
        "title": "Finale de la Coupe du monde",
        "url": "https://example.com/fifa-final-fr",
        "text": "La finale de la Coupe du monde de la FIFA est l’un des événements télévisés les plus regardés au monde.",
        "views": 150000000,
        "lang": "fr",
    },
    # Spanish
    {
        "title": "Final de la Copa Mundial",
        "url": "https://example.com/fifa-final-es",
        "text": "La final de la Copa Mundial de la FIFA es uno de los eventos televisados más vistos en el mundo.",
        "views": 150000000,
        "lang": "es",
    },
    # Italian
    {
        "title": "Finale della Coppa del Mondo",
        "url": "https://example.com/fifa-final-it",
        "text": "La finale della Coppa del Mondo FIFA è uno degli eventi televisivi più seguiti al mondo.",
        "views": 150000000,
        "lang": "it",
    },
    # Japanese
    {
        "title": "FIFAワールドカップ決勝",
        "url": "https://example.com/fifa-final-ja",
        "text": "FIFAワールドカップの決勝戦は、世界で最も視聴されるテレビ番組の一つです。",
        "views": 150000000,
        "lang": "ja",
    },
    # Arabic
    {
        "title": "نهائي كأس العالم",
        "url": "https://example.com/fifa-final-ar",
        "text": "نهائي كأس العالم FIFA يعد من أكثر الأحداث التلفزيونية مشاهدةً في العالم.",
        "views": 150000000,
        "lang": "ar",
    },
    # Chinese (Simplified)
    {
        "title": "FIFA 世界杯决赛",
        "url": "https://example.com/fifa-final-zh",
        "text": "FIFA 世界杯决赛是全球收视率最高的电视赛事之一。",
        "views": 150000000,
        "lang": "zh",
    },
    # Korean
    {
        "title": "FIFA 월드컵 결승전",
        "url": "https://example.com/fifa-final-ko",
        "text": "FIFA 월드컵 결승전은 전 세계에서 가장 많이 시청되는 TV 행사 중 하나입니다.",
        "views": 150000000,
        "lang": "ko",
    },
    # Hindi (you asked for code 'hii' — using exactly that to match your filter)
    {
        "title": "FIFA विश्व कप फाइनल",
        "url": "https://example.com/fifa-final-hii",
        "text": "FIFA विश्व कप का फाइनल दुनिया के सबसे अधिक देखे जाने वाले टेलीविज़न आयोजनों में से एक है।",
        "views": 150000000,
        "lang": "hii",
    },
]

for d in docs_multilang:
    client.data_object.create(d, "Articles")

# sanity check
print(client.query.aggregate("Articles").with_meta_count().do())

{'data': {'Aggregate': {'Articles': [{'meta': {'count': 23}}]}}}


# Keyword Search

In [59]:
def keyword_search(query,
                   results_lang='en',
                   properties = ["title","url","text"],
                   num_results=3):

    where_filter = {
    "path": ["lang"],
    "operator": "Equal",
    "valueString": results_lang
    }
    
    response = (
        client.query.get("Articles", properties)
        .with_bm25(
            query=query
        )
        .with_where(where_filter)
        .with_limit(num_results)
        .do()
        )

    result = response['data']['Get']['Articles']
    return result

schema = client.schema.get()
print(schema)

{'classes': [{'class': 'Articles', 'description': 'News and wiki articles', 'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2}, 'cleanupIntervalSeconds': 60, 'stopwords': {'additions': None, 'preset': 'en', 'removals': None}, 'usingBlockMaxWAND': True}, 'multiTenancyConfig': {'autoTenantActivation': False, 'autoTenantCreation': False, 'enabled': False}, 'properties': [{'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': True, 'name': 'title', 'tokenization': 'word'}, {'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': True, 'name': 'url', 'tokenization': 'word'}, {'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': True, 'name': 'text', 'tokenization': 'word'}, {'dataType': ['int'], 'indexFilterable': True, 'indexRangeFilters': False, 'indexSearchable': False, 'name': 'views'}, {'dataType': ['text'], 'indexFilterable': True, 'indexRangeFilters': False, 'ind

In [60]:
query = "What is the most viewed televised event?"
keyword_search_results = keyword_search(query)
print(keyword_search_results)

[{'text': 'The FIFA World Cup final is one of the most watched televised events worldwide.', 'title': 'FIFA World Cup Final', 'url': 'https://example.com/fifa-final-en'}, {'text': 'The FIFA World Cup final is one of the most watched televised events worldwide.', 'title': 'FIFA World Cup Final', 'url': 'https://example.com/fifa-final'}, {'text': 'The FIFA World Cup final is one of the most watched televised events worldwide.', 'title': 'FIFA World Cup Final', 'url': 'https://example.com/fifa-final-en'}]


In [61]:
properties = ["text", "title", "url", 
             "views", "lang"]

In [62]:
def print_result(result):
    """ Print results with colorful formatting """
    for i,item in enumerate(result):
        print(f'item {i}')
        for key in item.keys():
            print(f"{key}:{item.get(key)}")
            print()
        print()

In [63]:
print_result(keyword_search_results)
where_de = {"path": ["lang"], "operator": "Equal", "valueString": "de"}
print(client.query.aggregate("Articles").with_where(where_de).with_meta_count().do())
# meta count will be 0

item 0
text:The FIFA World Cup final is one of the most watched televised events worldwide.

title:FIFA World Cup Final

url:https://example.com/fifa-final-en


item 1
text:The FIFA World Cup final is one of the most watched televised events worldwide.

title:FIFA World Cup Final

url:https://example.com/fifa-final


item 2
text:The FIFA World Cup final is one of the most watched televised events worldwide.

title:FIFA World Cup Final

url:https://example.com/fifa-final-en


{'data': {'Aggregate': {'Articles': [{'meta': {'count': 2}}]}}}


## Try modifying the search options
- Other languages to try: `en, de, fr, es, it, ja, ar, zh, ko, hi`i

In [64]:
query = "What is the most viewed televised event?"
keyword_search_results = keyword_search(query, results_lang='de')
print_result(keyword_search_results)